Data Ingestion

In [3]:
###Document Structure

from langchain_core.documents import Document

In [4]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Krish Naik",
        "date_created":"2025-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Krish Naik', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [5]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [6]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [10]:
%pip install langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [langchain-community]ngchain-community]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
### TextLoader is a simple document loader that reads text files and creates Document objects. It takes the file path and encoding as input and returns a list of Document objects containing the content and metadata of the text file.
from langchain.document_loaders import TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

/Users/rohiniramasheshu/Documents/YTRAG/rag-project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [12]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [16]:
### Create a sample PDF file and load it using PyMuPDFLoader and do parse the text from it
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory using PyMuPDFLoader
dir_loader=DirectoryLoader(
    "../data/text_files/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'creationdate': '2026-04-03T21:13:17+00:00', 'source': '../data/text_files/pdf/objectdetection.pdf', 'file_path': '../data/text_files/pdf/objectdetection.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'objectdetection.pdf - Google Docs', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-03T21:13:17+00:00', 'trapped': '', 'modDate': "D:20260403211317+00'00'", 'creationDate': "D:20260403211317+00'00'", 'page': 0}, page_content="\u200bWhat is Retrieval-Augmented Generation?\u200b\n\u200bRetrieval-Augmented Generation (RAG) is the process of optimizing the output of a large\u200b\n\u200blanguage model, so it references an authoritative knowledge base outside of its training data\u200b\n\u200bsources before generating a response. Large Language Models (LLMs) are trained on vast\u200b\n\u200b

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Initialize the splitter
# chunk_size: characters per chunk (500-1000 is standard)
# chunk_overlap: keeps context between chunks (10-20% of size)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

# 2. Split the loaded documents into chunks
# This takes your list of Document objects and returns a larger list of smaller Document objects
final_chunks = text_splitter.split_documents(pdf_documents)

# 3. Verify the results
print(f"Original documents: {len(pdf_documents)}")
print(f"Created chunks: {len(final_chunks)}")
print(f"Sample content from first chunk:\n{final_chunks[0].page_content[:200]}...")



Original documents: 7
Created chunks: 17
Sample content from first chunk:
​What is Retrieval-Augmented Generation?​
​Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large​
​language model, so it references an authoritative knowledge base ou...


Embaddeing (converting text to vector) and vector store!